# Notebook 02: Functional Genomics & CRISPR Screen Analysis

This tutorial demonstrates:
1. Ingestion of pooled CRISPR guide RNA (sgRNA) count data.
2. Median-of-ratios library size normalization.
3. Log2 Fold Change (LFC) calculation across Day 0 (baseline) vs Day 21 (endpoint).
4. Gene-level essentiality testing with Robust Rank Aggregation (RRA).
5. Perturbomics-style amino-acid-level pocket scoring for top hits.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from functional_genomics.crispr_screen import CRISPRScreenAnalyzer
from functional_genomics.perturbomics import PerturbomicsPocketFinder
from utils.viz_utils import plot_volcano

## 1. Simulate Pooled CRISPRi Screen Data

We simulate a focused screen targeting candidate disease genes with 4 sgRNAs per gene.

In [ ]:
np.random.seed(42)
genes = ["FLT3", "NTRK1", "GLP1R", "TCF7L2", "PPARG", "IDH1", "CONTROL_NON_ESS"] * 4
sgrna_ids = [f"sgRNA_{i:03d}" for i in range(len(genes))]

# Base counts at Day 0
counts_t0 = np.random.negative_binomial(n=50, p=0.08, size=len(genes))

# Depletion in oncology/essential targets
depletion_factors = {
    "FLT3": 0.15, "IDH1": 0.20, "NTRK1": 0.35,
    "GLP1R": 0.90, "TCF7L2": 0.85, "PPARG": 0.95, "CONTROL_NON_ESS": 1.05
}
counts_t21 = [int(counts_t0[i] * depletion_factors[genes[i]] * np.random.uniform(0.85, 1.15)) for i in range(len(genes))]

screen_df = pd.DataFrame({
    "sgRNA": sgrna_ids,
    "GENE": genes,
    "T0_COUNT": counts_t0,
    "T21_COUNT": counts_t21
})
screen_df.head(10)

## 2. Size-Factor Normalization and Log2 Fold Change

In [ ]:
analyzer = CRISPRScreenAnalyzer()
norm_df = analyzer.normalize_counts(screen_df, sample_cols=["T0_COUNT", "T21_COUNT"])
lfc_df = analyzer.compute_sgrna_fold_change(norm_df, control_col="T0_COUNT_NORM", treatment_col="T21_COUNT_NORM")
lfc_df.head()

## 3. Aggregate Gene Essentiality and Rank Vulnerabilities

In [ ]:
gene_summary = analyzer.aggregate_gene_essentiality(lfc_df)
print("Ranked Gene Essentiality:")
gene_summary

## 4. Volcano Plot of Vulnerabilities

In [ ]:
fig = plot_volcano(gene_summary, fdr_threshold=0.05, lfc_threshold=-0.5)
plt.show()

## 5. Perturbomics: Amino-Acid-Level Druggable Pocket Discovery

In [ ]:
finder = PerturbomicsPocketFinder(min_pocket_residues=4, druggability_threshold=0.60)
mock_tiling = pd.DataFrame({
    "RESIDUE_NUM": list(range(570, 595)),
    "AA": ["VAL", "LEU", "PHE", "ARG", "TRP", "ILE", "TYR", "GLU", "ASP", "ALA"] * 3,
    "FITNESS_EFFECT": [-2.8, -3.1, -2.6, -1.9, -2.9, -2.7, -0.4, -0.2, -0.5, -0.1] * 3
}).iloc[:25]

scored = finder.calculate_residue_druggability(mock_tiling)
pockets = finder.find_pockets(scored)
print(f"Discovered {len(pockets)} high-vulnerability pocket(s):")
for p in pockets:
    print(f" - {p['POCKET_ID']}: Residues {p['START_RESIDUE']}-{p['END_RESIDUE']}, Druggability: {p['DRUGGABILITY_SCORE']}")